In [1]:
import numpy as np
import pandas as pd
import sklearn


In [2]:
df_train = pd.read_csv('../data/processed/train_data.csv')
df_test = pd.read_csv('../data/processed/test_data.csv')

In [4]:
df_test.head(1)

,year,month,day,hour,PM2.5,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,PM2.5_shift1,PM10_shift1,SO2_shift1,NO2_shift1,CO_shift1,O3_shift1
0,2017,1,1,0,485.0,-4.7,1022.1,-6.1,0.0,ENE,1.0,Aotizhongxin,472.0,504.0,9.0,140.0,5400.0,3.0


In [5]:
X_train = df_train.drop(columns=['PM2.5'])
y_train = df_train['PM2.5']
X_test = df_test.drop(columns=['PM2.5'])
y_test = df_test['PM2.5']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

cat_col = ['station','wd']
num_col = [col for col in X_train.columns if col not in cat_col] 

processing = ColumnTransformer(transformers=[
    ('encode',OneHotEncoder(drop='first',sparse_output=False),cat_col),
    ('scale',StandardScaler(),num_col)
])

In [7]:
X_train_processed = processing.fit_transform(X_train)
X_test_processed = processing.transform(X_test)

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [21]:
from sklearn.metrics import mean_squared_error, r2_score,root_mean_squared_error
import time

import warnings
warnings.filterwarnings('ignore')

In [18]:
modellar = {
    'Liner Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'ElasticNet Regression': ElasticNet(),
    'SGD Regression': SGDRegressor(),
    'Decision Tree Regression': DecisionTreeRegressor(max_depth=15),
    'Linear SVR': LinearSVR(),
    'KNN Regression': KNeighborsRegressor(),
    'Random Forest Regression': RandomForestRegressor(n_estimators=70,n_jobs=-1,max_depth=15),
    'Extra Trees Regression': ExtraTreesRegressor(n_estimators=70,n_jobs=-1,max_depth=15),
    'Hist Gradient Boosting Regression': HistGradientBoostingRegressor(),
    'MLP Regression': MLPRegressor(max_iter=500),
    
    'XGBoost Regression': XGBRegressor(),
    'LightGBM Regression': LGBMRegressor(),
    'CatBoost Regression': CatBoostRegressor(verbose=0)
}

In [ ]:
natijalar = []

for nomi, model in modellar.items():
    print(f"⏳ {nomi} o'qitilmoqda...")
    start_time = time.time()
    
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    lag_time = time.time() - start_time
    
    natijalar.append({
        'Model': nomi,
        'MSE': round(mse, 3),
        'RMSE': round(rmse, 3),
        'R2 Score': round(r2, 3),
        'Ketgan vaqt (sek)': round(lag_time, 3)
    })
    
df_natija = pd.DataFrame(natijalar)

⏳ Liner Regression o'qitilmoqda...
⏳ Ridge Regression o'qitilmoqda...
⏳ Lasso Regression o'qitilmoqda...
⏳ ElasticNet Regression o'qitilmoqda...
⏳ SGD Regression o'qitilmoqda...
⏳ Decision Tree Regression o'qitilmoqda...
⏳ Linear SVR o'qitilmoqda...
⏳ KNN Regression o'qitilmoqda...
⏳ Random Forest Regression o'qitilmoqda...
⏳ Extra Trees Regression o'qitilmoqda...
⏳ Hist Gradient Boosting Regression o'qitilmoqda...
⏳ MLP Regression o'qitilmoqda...
⏳ XGBoost Regression o'qitilmoqda...
⏳ LightGBM Regression o'qitilmoqda...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005841 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2778
[LightGBM] [Info] Number of data points in the train set: 403764, number of used features: 41
[LightGBM] [Info] Start training from score 79.221276


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


⏳ CatBoost Regression o'qitilmoqda...
                                Model       MSE    RMSE  R2 Score  \
14                CatBoost Regression   612.359  24.746     0.952   
11                     MLP Regression   620.191  24.904     0.951   
10  Hist Gradient Boosting Regression   632.534  25.150     0.950   
9              Extra Trees Regression   629.970  25.099     0.950   
13                LightGBM Regression   629.663  25.093     0.950   
8            Random Forest Regression   648.361  25.463     0.949   
12                 XGBoost Regression   665.019  25.788     0.948   
0                    Liner Regression   677.614  26.031     0.947   
1                    Ridge Regression   677.612  26.031     0.947   
2                    Lasso Regression   691.808  26.302     0.946   
4                      SGD Regression   683.400  26.142     0.946   
6                          Linear SVR   698.505  26.429     0.945   
5            Decision Tree Regression  1009.726  31.776     0.921

In [20]:
df_natija.sort_values(by='R2 Score', ascending=False)

,Model,MSE,RMSE,R2 Score,Ketgan vaqt (sek)
14,CatBoost Regression,612.359,24.746,0.952,19.744
11,MLP Regression,620.191,24.904,0.951,456.781
10,Hist Gradient Boosting Regression,632.534,25.150,0.950,3.109
9,Extra Trees Regression,629.970,25.099,0.950,65.980
13,LightGBM Regression,629.663,25.093,0.950,0.937
8,Random Forest Regression,648.361,25.463,0.949,46.053
12,XGBoost Regression,665.019,25.788,0.948,1.205
0,Liner Regression,677.614,26.031,0.947,0.737
1,Ridge Regression,677.612,26.031,0.947,0.192
2,Lasso Regression,691.808,26.302,0.946,0.310


In [23]:
from sklearn.model_selection import cross_validate,TimeSeriesSplit

tsv = TimeSeriesSplit(n_splits=5)

natija = cross_validate(HistGradientBoostingRegressor(), X_train_processed, y_train, cv=tsv, scoring='r2') 
natija['test_score']


array([0.94820699, 0.95101392, 0.95170755, 0.94931038, 0.95160804])

In [27]:
# Qaysi ustun PM2.5 ni topishda eng katta "shpargalka" vazifasini bajaryapti?
korrelyatsiya = df_train.corrwith(df_train['PM2.5'],numeric_only=True).sort_values(ascending=False)
korrelyatsiya

PM2.5           1.000000
PM2.5_shift1    0.969408
PM10_shift1     0.860629
CO_shift1       0.775095
NO2_shift1      0.662505
SO2_shift1      0.479056
DEWP            0.115675
month           0.022434
PRES            0.021479
day             0.021268
hour            0.013900
RAIN           -0.014395
year           -0.045224
TEMP           -0.130376
O3_shift1      -0.144822
WSPM           -0.266813
dtype: float64